In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('train.csv')



In [3]:
df.isnull().sum()

cust_id                                 0
age                                     0
Gender                                  0
Usage Frequency                         0
Support Calls                           0
type_plan                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
Total Spend                             0
no_of_special_requests                  0
booking_status                          0
dtype: int64

In [10]:
df.tail()

,cust_id,age,Gender,Usage Frequency,Support Calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total Spend,no_of_special_requests,booking_status
29015,INN16851,44,Female,5,5,Premium,2023,12,26,Offline,0,0,471,2,Not_Canceled
29016,INN06266,61,Female,8,3,Standard,2023,10,16,Online,0,0,760,0,Canceled
29017,INN11285,34,Male,2,8,Basic,2024,5,24,Corporate,0,0,443,1,Not_Canceled
29018,INN00861,18,Male,10,0,Standard,2024,6,7,Online,0,0,482,0,Canceled
29019,INN15796,53,Female,2,5,Standard,2024,9,15,Online,0,0,475,0,Not_Canceled


In [3]:
## Dropping 'cust_id' as it is not useful for our analysis beacsue it is unique for each customer
df.drop(columns=['cust_id'], inplace=True)
df.head()

,age,Gender,Usage Frequency,Support Calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total Spend,no_of_special_requests,booking_status
0,51,Male,26,10,Premium,2023,10,17,Online,0,0,549,0,Not_Canceled
1,34,Male,9,8,Premium,2024,7,16,Online,0,0,308,2,Not_Canceled
2,60,Male,8,5,Standard,2024,9,8,Offline,0,0,828,0,Canceled
3,57,Female,4,0,Basic,2024,8,8,Offline,0,0,314,0,Not_Canceled
4,40,Male,13,0,Standard,2024,6,15,Offline,0,0,239,0,Canceled


In [4]:
df.rename(columns={'Support Calls': 'Support_calls','Usage Frequency':'Usage_frequency','Total Spend':'Total_spend'}, inplace=True)

In [9]:
df.head(1)

,age,Gender,Usage_frequency,Support_calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total_spend,no_of_special_requests,booking_status
0,51,Male,26,10,Premium,2023,10,17,Online,0,0,549,0,Not_Canceled


In [5]:
cat_cols = [
    'booking_status',
    'Gender',
    'type_plan',
    'market_segment_type']

num_cols = [
    'age',
    'Usage_frequency',
    'Support_calls',
    'arrival_year',
    'arrival_month',
    'arrival_date',
    'no_of_previous_cancellations',
    'no_of_previous_bookings_not_canceled',
    'no_of_special_requests',
    'Total_spend',
]

In [6]:
df_copy = df.copy()

## Univariate Analysis

In [15]:
df_copy.head(1)

,age,Gender,Usage_frequency,Support_calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total_spend,no_of_special_requests,booking_status
0,51,Male,26,10,Premium,2023,10,17,Online,0,0,549,0,Not_Canceled


In [7]:
##### Label encoding
from sklearn.preprocessing import LabelEncoder

In [8]:
label_encoder = LabelEncoder()

mappings={}

for col in cat_cols:
    df[col] = label_encoder.fit_transform(df[col])

    mappings[col] = {label:code for label,code in zip(label_encoder.classes_ , label_encoder.transform(label_encoder.classes_))}

In [10]:
!pip install statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.8 MB 4.2 MB/s eta 0:00:03
   ------- -------------------------------- 1.8/9.8 MB 4.8 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.8 MB 3.8 MB/s eta 0:00:02
   ----------- ---------------------------- 2.9/9.8 MB 3.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.7/9.8 MB 3.6 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.8 MB 4.2 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.8 MB 4.2 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.8 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 7.6/9.8 MB 4.1 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.8 MB 4.2 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.8 MB 3.9 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 3.8 MB/s eta 0:00:01
   ----------------

In [11]:
X = add_constant(df)

vif_data = pd.DataFrame()

vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values,i) for i in range(X.shape[1])]

In [12]:
vif_data

,feature,VIF
0,const,3.365580e+07
1,age,1.000471e+00
2,Gender,1.000791e+00
3,Usage_frequency,1.000338e+00
4,Support_calls,1.000615e+00
5,type_plan,1.000859e+00
6,arrival_year,1.210299e+00
7,arrival_month,1.166947e+00
8,arrival_date,1.003016e+00
9,market_segment_type,1.243560e+00


In [13]:
corr = df.corr()
corr

,age,Gender,Usage_frequency,Support_calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total_spend,no_of_special_requests,booking_status
age,1.000000,0.004748,0.000865,-0.005531,0.003925,-0.002059,-0.001225,-0.008485,0.006201,-0.007006,-0.004567,-0.013649,-0.004997,-0.002697
Gender,0.004748,1.000000,0.002196,0.010472,-0.013609,-0.007967,-0.008943,0.004343,-0.005229,-0.002392,0.004088,-0.011652,-0.009313,-0.001469
Usage_frequency,0.000865,0.002196,1.000000,0.000645,-0.000101,-0.007909,0.010661,-0.000229,-0.001647,-0.002821,0.007964,0.006639,-0.003987,0.000997
Support_calls,-0.005531,0.010472,0.000645,1.000000,-0.006837,-0.001014,0.011057,-0.001366,-0.004631,0.000860,0.001480,-0.012529,0.009370,0.005691
type_plan,0.003925,-0.013609,-0.000101,-0.006837,1.000000,-0.002490,0.017378,0.000524,-0.007449,0.008899,0.002009,0.000075,0.010476,0.005544
arrival_year,-0.002059,-0.007967,-0.007909,-0.001014,-0.002490,1.000000,-0.339353,0.016373,0.149341,0.005358,0.026980,-0.005804,0.058387,-0.173351
arrival_month,-0.001225,-0.008943,0.010661,0.011057,0.017378,-0.339353,1.000000,-0.043967,-0.009681,-0.037577,-0.005398,0.005473,0.108712,0.012487
arrival_date,-0.008485,0.004343,-0.000229,-0.001366,0.000524,0.016373,-0.043967,1.000000,0.011523,-0.010674,-0.000837,0.007411,0.018331,-0.008020
market_segment_type,0.006201,-0.005229,-0.001647,-0.004631,-0.007449,0.149341,-0.009681,0.011523,1.000000,-0.075912,-0.204948,-0.004989,0.307481,-0.136848
no_of_previous_cancellations,-0.007006,-0.002392,-0.002821,0.000860,0.008899,0.005358,-0.037577,-0.010674,-0.075912,1.000000,0.475414,-0.001819,-0.000653,0.032081


In [14]:
skewness  = df.skew()
skewness

age                                     -0.005832
Gender                                  -0.003170
Usage_frequency                         -0.003017
Support_calls                           -0.019948
type_plan                               -0.009996
arrival_year                            -1.669774
arrival_month                           -0.346711
arrival_date                             0.027335
market_segment_type                     -1.670290
no_of_previous_cancellations            24.940984
no_of_previous_bookings_not_canceled    19.509733
Total_spend                             -0.016122
no_of_special_requests                   1.150448
booking_status                          -0.741025
dtype: float64

In [17]:
for col in df.columns:
        df[col] = np.log1p(df[col])

In [18]:
df.head(1)

,age,Gender,Usage_frequency,Support_calls,type_plan,arrival_year,arrival_month,arrival_date,market_segment_type,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,Total_spend,no_of_special_requests,booking_status
0,3.951244,0.693147,3.295837,2.397895,0.693147,7.612831,2.397895,2.890372,1.609438,0.0,0.0,6.309918,0.0,0.693147
